# Teacher Pilot — Qwen3-30B-A3B-Instruct-2507

Calibrates the synthetic-data generation cost budgeted in proposal §4 (4.0 H200-hrs for 800 datasets).

**Workflow:**
1. Pull N popular datasets from `data.wa.gov` via Socrata Catalog API.
2. For each, fetch 20 sample rows + schema, build the teacher prompt.
3. Run inference with vLLM on a single H200.
4. Extrapolate per-dataset latency to the full 800-dataset run.

**Prereqs (on the H200 host):**

In [ ]:
%pip install vllm pandas

## 1. Configuration

In [ ]:
import json
import re
import time
import urllib.request
from pathlib import Path
from statistics import mean, median

import pandas as pd

TEACHER_MODEL = "Qwen/Qwen3-30B-A3B-Instruct-2507"
DOMAIN = "data.wa.gov"
CATALOG_URL = f"https://{DOMAIN}/api/catalog/v1"

NUM_DATASETS = 5  # pilot size — bump to 10 for lower-variance estimate
SAMPLE_ROWS = 20  # rows per dataset to include in the prompt
MAX_OUTPUT_TOKENS = 6000  # ceiling per generated datacard
MAX_MODEL_LEN = 32768  # vLLM context window

FULL_RUN_DATASETS = 800  # target for extrapolation (proposal §4)
BUDGETED_HOURS = 4.0  # what we promised in the proposal

## 2. Teacher prompt

In [ ]:
SYSTEM_PROMPT = (
    "You are a precise technical writer producing dataset documentation. "
    "Use only information explicitly present in the provided metadata. "
    "If a field is missing, write 'Not provided in source metadata.' "
    "Do not speculate, do not editorialize, and do not invent statistics or "
    "legal context. Start sentences directly with facts."
)

USER_PROMPT_TEMPLATE = """\
Generate a dataset card for the following dataset, following the Google Data
Cards Playbook structure. Output Markdown only, no preamble.

Required sections (in order):
1. # Dataset Card: <title>
2. ## Quick Facts (publisher, category, update frequency, size, dates, license)
3. ## Dataset Overview (1 paragraph, grounded in the metadata)
4. ## Data Dictionary (table of column name | type | description)
5. ## Context, Provenance, & Limitations (legal restrictions, schema evolution, reliability)

Input metadata (JSON):
```json
{payload}
```
"""

## 3. Fetch popular datasets from data.wa.gov

In [ ]:
def strip_html(s):
    return re.sub(r"<[^>]+>", "", s or "").strip()


def fetch_popular_datasets(n):
    """Top-N most-viewed tabular datasets from data.wa.gov."""
    url = (
        f"{CATALOG_URL}?domains={DOMAIN}&search_context={DOMAIN}"
        f"&only=datasets&order=page_views_last_month"
        f"&limit={n * 4}"  # over-fetch to allow filtering
    )
    with urllib.request.urlopen(url, timeout=60) as resp:
        data = json.load(resp)

    picks = []
    for r in data.get("results", []):
        res = r.get("resource", {}) or {}
        if res.get("type") != "dataset":
            continue
        if len(res.get("columns_name") or []) < 3:
            continue
        picks.append(r)
        if len(picks) >= n:
            break
    return picks


entries = fetch_popular_datasets(NUM_DATASETS)
print(f"Selected {len(entries)} datasets:")
for e in entries:
    r = e["resource"]
    print(f"  • {r['id']}  {r['name'][:70]}")

## 4. Build payloads + prompts

In [ ]:
def fetch_sample_rows(dataset_uid):
    url = f"https://{DOMAIN}/resource/{dataset_uid}.json?$limit={SAMPLE_ROWS}"
    with urllib.request.urlopen(url, timeout=60) as resp:
        rows = json.load(resp)
    return pd.DataFrame(rows) if rows else pd.DataFrame()


def build_schema_info(df, column_names, column_types):
    info = []
    for name, sodata_type in zip(column_names, column_types):
        col = {"column": name, "type": sodata_type}
        if name not in df.columns:
            info.append(col)
            continue
        series = df[name]
        col["non_null"] = int(series.notna().sum())
        numeric = pd.to_numeric(series, errors="coerce")
        if numeric.notna().sum() > 0 and sodata_type.lower() in {
            "number",
            "money",
            "double",
            "float",
        }:
            col["min"] = float(numeric.min())
            col["max"] = float(numeric.max())
            col["mean"] = round(float(numeric.mean()), 2)
        else:
            uniques = series.dropna().astype(str).unique().tolist()[:5]
            if uniques:
                col["unique_samples"] = uniques
        info.append(col)
    return info


def build_payload(catalog_entry):
    res = catalog_entry.get("resource", {}) or {}
    cls = catalog_entry.get("classification", {}) or {}
    uid = res.get("id")
    columns_name = res.get("columns_name") or []
    columns_type = res.get("columns_datatype") or []
    df = fetch_sample_rows(uid)
    license_field = res.get("license")
    license_name = (
        license_field.get("name") if isinstance(license_field, dict) else None
    )
    return {
        "dataset_title": res.get("name"),
        "dataset_uid": uid,
        "publisher": res.get("attribution") or "Not provided in source metadata.",
        "category": cls.get("domain_category") or "Uncategorized",
        "tags": cls.get("domain_tags") or [],
        "license": license_name,
        "created_at": res.get("createdAt"),
        "updated_at": res.get("updatedAt"),
        "global_description": strip_html(res.get("description")),
        "schema_info": build_schema_info(df, columns_name, columns_type),
        "sample_rows": (
            df.head(SAMPLE_ROWS).to_dict(orient="records") if not df.empty else []
        ),
    }


def build_prompt(payload):
    return USER_PROMPT_TEMPLATE.format(
        payload=json.dumps(payload, indent=2, default=str)
    )


payloads = [build_payload(e) for e in entries]
prompts = [build_prompt(p) for p in payloads]

for p, prompt in zip(payloads, prompts):
    print(
        f"  • {p['dataset_uid']}  {p['dataset_title'][:50]:<50}  "
        f"{len(p['schema_info']):>3} cols, {len(p['sample_rows']):>3} rows, "
        f"{len(prompt):>7,} chars"
    )

### Inspect a sample prompt (optional)

In [ ]:
print(prompts[0][:3000])
print("\n[…truncated…]")

## 5. Load Qwen3-30B-A3B with vLLM

First time the model is loaded, weights are downloaded from the HF Hub (~60 GB BF16). Subsequent loads pull from local cache.

In [ ]:
from vllm import LLM, SamplingParams

import sys, os

sys.stdout = open(os.dup(1), "w", buffering=1)
sys.stderr = open(os.dup(2), "w", buffering=1)

load_start = time.perf_counter()
llm = LLM(
    model=TEACHER_MODEL,
    dtype="bfloat16",
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=0.90,
    trust_remote_code=True,
)
load_secs = time.perf_counter() - load_start
tokenizer = llm.get_tokenizer()
print(f"Model loaded in {load_secs:.1f}s")

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

## 6. Run inference

In [ ]:
chat_prompts = [
    tokenizer.apply_chat_template(
        [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": p},
        ],
        tokenize=False,
        add_generation_prompt=True,
    )
    for p in prompts
]

sampling = SamplingParams(temperature=0.3, top_p=0.9, max_tokens=MAX_OUTPUT_TOKENS)

gen_start = time.perf_counter()
outputs = llm.generate(chat_prompts, sampling)
gen_secs = time.perf_counter() - gen_start

per_prompt = []
for chat, out in zip(chat_prompts, outputs):
    completion = out.outputs[0]
    per_prompt.append(
        {
            "input_tokens": len(tokenizer.encode(chat)),
            "output_tokens": len(completion.token_ids),
            "text": completion.text,
        }
    )

print(f"Generation wall time: {gen_secs:.1f}s for {len(prompts)} datasets")

## 7. Summary + extrapolation to full run

In [ ]:
in_tokens = [p["input_tokens"] for p in per_prompt]
out_tokens = [p["output_tokens"] for p in per_prompt]
total_out = sum(out_tokens)
total_in = sum(in_tokens)

n = len(per_prompt)
secs_per_dataset = gen_secs / n
output_tok_per_sec = total_out / gen_secs if gen_secs > 0 else 0
extrapolated_hours = (secs_per_dataset * FULL_RUN_DATASETS) / 3600

print("=" * 64)
print(f"Pilot results ({n} datasets)")
print("=" * 64)
print(f"Total generation wall time : {gen_secs:.1f}s")
print(
    f"Total input tokens         : {total_in:,} (mean {mean(in_tokens):.0f}, "
    f"median {median(in_tokens):.0f}, max {max(in_tokens):,})"
)
print(
    f"Total output tokens        : {total_out:,} (mean {mean(out_tokens):.0f}, "
    f"median {median(out_tokens):.0f}, max {max(out_tokens):,})"
)
print(f"Throughput (output)        : {output_tok_per_sec:.1f} tok/s aggregate")
print(f"Seconds per dataset        : {secs_per_dataset:.1f}s")
print()
print(f"Extrapolated full run ({FULL_RUN_DATASETS} datasets):")
print(f"  ≈ {extrapolated_hours:.1f} H200-hours")
print(f"  (proposal §4 allocates {BUDGETED_HOURS} H200-hrs)")
if extrapolated_hours > BUDGETED_HOURS:
    print(
        f"  ⚠ exceeds proposal budget by {extrapolated_hours - BUDGETED_HOURS:.1f} hrs — revise §4 row"
    )
else:
    print(
        f"  ✓ within proposal budget (slack: {BUDGETED_HOURS - extrapolated_hours:.1f} hrs)"
    )

## 8. Inspect one generated datacard

In [ ]:
from IPython.display import Markdown, display

display(Markdown(f"### {payloads[0]['dataset_title']} — {payloads[0]['dataset_uid']}"))
display(Markdown(per_prompt[0]["text"]))

## 9. Save full results

In [ ]:
output_path = Path("pilot_results.json")
output_path.write_text(
    json.dumps(
        {
            "model": TEACHER_MODEL,
            "load_secs": load_secs,
            "gen_secs": gen_secs,
            "datasets": [
                {**payload, "generation": prompt_result}
                for payload, prompt_result in zip(payloads, per_prompt)
            ],
        },
        indent=2,
        default=str,
    )
)
print(f"Wrote {output_path}")